# Grid Search: Clustering Threshold & Post-Processing Parameters

**Goal:** Systematically find optimal hyperparameters using train data as a dev set.

**Strategy:**
1. Phase 1: Sweep clustering `threshold` (biggest single knob)
2. Phase 2: Sweep `min_duration_off` alongside best threshold
3. Phase 3: Sweep post-processing params (`min_duration`, `merge_gap`, relabel params)

Uses Demucs-preprocessed vocals + the same custom Bengali segmentation model.

In [ ]:
# Force compatible numpy + scipy
!pip install numpy==1.26.4 scipy==1.11.4 --force-reinstall

# Then install pyannote + lightning
!pip install pyannote.audio==4.0.1 lightning>=2.4

# Demucs for vocal separation
!pip install -q demucs

In [ ]:
import gc, json, csv, itertools, time
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import torch
import torchaudio
from tqdm.auto import tqdm

from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## Paths & Data Loading

Load train annotations. There are only 10 training files — we use **all of them** as the dev set.
CSV format: 3 columns (`start_time`, `end_time`, `speaker_id`) with `HH:MM:SS` timestamps.

In [ ]:
# === PATHS ===
BASE = Path("/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization")
TRAIN_AUDIO_DIR = BASE / "train" / "audio"
TRAIN_ANNOT_DIR = BASE / "train" / "annotation"
VOCALS_DIR = Path("/kaggle/working/vocals_train")
VOCALS_DIR.mkdir(parents=True, exist_ok=True)

# === Discover all train files ===
annot_paths = sorted(TRAIN_ANNOT_DIR.glob("*.csv"))
audio_paths = sorted(TRAIN_AUDIO_DIR.glob("*.wav"))

# Build stem → paths mapping
annot_stems = {p.stem: p for p in annot_paths}
audio_stems = {p.stem: p for p in audio_paths}
common_stems = sorted(set(annot_stems) & set(audio_stems))

print(f"Train annotations: {len(annot_paths)}")
print(f"Train audio files: {len(audio_paths)}")
print(f"Matched pairs:     {len(common_stems)}")

# === Use ALL files as dev set (only 10 available) ===
dev_stems = common_stems
print(f"\nDev set: ALL {len(dev_stems)} files (no subsetting needed)")

In [ ]:
def parse_hhmmss(t: str) -> float:
    """Convert HH:MM:SS or MM:SS string to seconds."""
    t = str(t).strip()
    parts = t.split(":")
    if len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    elif len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    return float(parts[0])


def load_annotation_csv(csv_path: Path) -> List[Dict]:
    """Load annotation CSV — matches the working notebook's approach exactly.
    
    Uses plain pd.read_csv() + df.columns.str.strip().
    """
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    segments = []
    for _, row in df.iterrows():
        start = parse_hhmmss(row["start_time"])
        end = parse_hhmmss(row["end_time"])
        spk_id = f"SPEAKER_{str(row['speaker_id']).strip()}"
        if end > start:
            segments.append({
                "start_time": start,
                "end_time": end,
                "speaker_id": spk_id,
            })
    return segments


def segments_to_pyannote(segments: List[Dict], uri: str = "") -> Annotation:
    """Convert segment dicts → pyannote Annotation for DER computation."""
    ann = Annotation(uri=uri)
    for seg in segments:
        ann[Segment(seg["start_time"], seg["end_time"])] = seg["speaker_id"]
    return ann


# Load ground truth for dev set — skip files that fail (e.g. train_013)
ground_truth = {}
skipped = []
for stem in dev_stems:
    csv_path = annot_stems[stem]
    try:
        if stem == dev_stems[0]:
            df_debug = pd.read_csv(csv_path)
            df_debug.columns = df_debug.columns.str.strip()
            print(f"DEBUG — Columns: {list(df_debug.columns)}")
            print(df_debug.head())
        segs = load_annotation_csv(csv_path)
        ground_truth[stem] = segments_to_pyannote(segs, uri=stem)
    except Exception as e:
        print(f"⚠ Skipping {stem}: {e}")
        skipped.append(stem)

# Remove skipped stems from dev_stems so subsequent cells don't try them
if skipped:
    dev_stems = [s for s in dev_stems if s not in skipped]
    print(f"\nSkipped {len(skipped)} files: {skipped}")

print(f"\nLoaded ground truth for {len(ground_truth)} files")

# Show stats for a sample
sample_stem = dev_stems[0]
sample_ann = ground_truth[sample_stem]
print(f"\nSample: {sample_stem}")
print(f"  Speakers: {len(sample_ann.labels())}")
print(f"  Segments: {len(list(sample_ann.itertracks()))}")
print(f"  Duration: {sample_ann.get_timeline().extent().end:.1f}s")

## Demucs Vocal Separation (Dev Set Only)

Run Demucs on the dev subset only to save time. Cache results to disk.

In [ ]:
from demucs.pretrained import get_model
from demucs.apply import apply_model

print("Loading Demucs htdemucs …")
demucs_model = get_model("htdemucs")
demucs_model.eval()
if torch.cuda.is_available():
    demucs_model = demucs_model.cuda()
demucs_sr = demucs_model.samplerate  # 44100

print(f"Processing {len(dev_stems)} dev files …\n")

for stem in tqdm(dev_stems, desc="Demucs (dev)"):
    out_path = VOCALS_DIR / f"{stem}.wav"
    if out_path.exists():
        continue

    wav_path = audio_stems[stem]
    waveform, orig_sr = torchaudio.load(wav_path)

    if orig_sr != demucs_sr:
        waveform = torchaudio.transforms.Resample(orig_sr, demucs_sr)(waveform)
    if waveform.shape[0] == 1:
        waveform = waveform.repeat(2, 1)

    with torch.no_grad():
        sources = apply_model(
            demucs_model,
            waveform.unsqueeze(0).cuda(),
            device="cuda",
            split=True,
            overlap=0.25,
            progress=False,
        )

    vocals = sources[0, 3].cpu().mean(dim=0)
    if orig_sr != demucs_sr:
        vocals = torchaudio.transforms.Resample(demucs_sr, orig_sr)(vocals.unsqueeze(0)).squeeze(0)

    torchaudio.save(str(out_path), vocals.unsqueeze(0), orig_sr)
    del waveform, sources, vocals
    gc.collect(); torch.cuda.empty_cache()

# Free Demucs
del demucs_model
gc.collect(); torch.cuda.empty_cache()

print(f"\n✅ Vocals ready: {len(list(VOCALS_DIR.glob('*.wav')))} files in {VOCALS_DIR}")

## Load Pyannote Pipeline (same as best notebook)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    huggingface_token = user_secrets.get_secret("hugging_face")
    login(token=huggingface_token)
    print("Hugging Face login successful!")
except Exception as e:
    print(f"Hugging Face login failed: {e}")

In [ ]:
import torch.serialization
from pyannote.audio.core.task import Specifications, Problem, Resolution
from pyannote.audio import Pipeline
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

torch.serialization.add_safe_globals([Specifications, Problem, Resolution])

# Load base pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1")

# Load custom Bengali segmentation weights
safetensors_path = hf_hub_download(
    repo_id="lucius-40/speaker-segmentation-fine-tuned-bn",
    filename="model.safetensors"
)
custom_weights = load_file(safetensors_path)
custom_weights = {k.replace("model.", "", 1): v for k, v in custom_weights.items()}
pipeline._segmentation.model.load_state_dict(custom_weights)

pipeline.to(DEVICE)
print("Pipeline loaded with custom Bengali segmentation model")

## Helper: Diarize & Evaluate

Core functions: run pipeline with given params → compute DER against ground truth.
Uses `pyannote.metrics` with configurable collar (check competition rules — typically 0.25s).

In [ ]:
# === DER collar: check competition rules. Common values: 0.0, 0.25, 0.5 ===
# If the competition does NOT specify a collar, use 0.0 (strictest).
# pyannote default collar=0.25. Adjust if you know the competition value.
COLLAR = 0.25

# ── Post-processing functions (same as best notebook) ───────────────

def fix_overlaps(segments):
    if not segments:
        return segments, 0, 0
    segments = sorted(segments, key=lambda x: x['start_time'])
    fixed, overlap_count, removed_count = [], 0, 0
    for i, seg in enumerate(segments):
        current = seg.copy()
        if i > 0 and fixed and fixed[-1]['end_time'] > current['start_time']:
            overlap_count += 1
            current['start_time'] = fixed[-1]['end_time']
        if current['end_time'] - current['start_time'] >= 0.01:
            fixed.append(current)
        else:
            removed_count += 1
    return fixed, overlap_count, removed_count


def relabel_isolated_segments(segments, max_duration=0.15, max_gap=0.3):
    if len(segments) < 3:
        return segments, 0
    segments = sorted(segments, key=lambda x: x['start_time'])
    relabeled, count = [], 0
    for i in range(len(segments)):
        current = segments[i].copy()
        if 0 < i < len(segments) - 1:
            prev_seg, next_seg = segments[i - 1], segments[i + 1]
            dur = current['end_time'] - current['start_time']
            gap_b = current['start_time'] - prev_seg['end_time']
            gap_a = next_seg['start_time'] - current['end_time']
            if (dur < max_duration and prev_seg['speaker_id'] == next_seg['speaker_id']
                    and current['speaker_id'] != prev_seg['speaker_id']
                    and gap_b < max_gap and gap_a < max_gap):
                current['speaker_id'] = prev_seg['speaker_id']
                count += 1
        relabeled.append(current)
    return relabeled, count


def filter_short_segments(segments, min_duration):
    filtered, removed = [], 0
    for seg in segments:
        if seg['end_time'] - seg['start_time'] >= min_duration:
            filtered.append(seg)
        else:
            removed += 1
    return filtered, removed


def merge_same_speaker(segments, merge_gap):
    if not segments:
        return segments, 0
    segments = sorted(segments, key=lambda x: x['start_time'])
    merged, count = [], 0
    current = segments[0].copy()
    for seg in segments[1:]:
        gap = seg['start_time'] - current['end_time']
        if current['speaker_id'] == seg['speaker_id'] and gap < merge_gap:
            current['end_time'] = seg['end_time']
            count += 1
        else:
            merged.append(current)
            current = seg.copy()
    merged.append(current)
    return merged, count


def postprocess(segments, min_duration, merge_gap, relabel_max_dur=None, relabel_max_gap=0.3):
    """Full post-processing pipeline with iterative relabel+merge."""
    if relabel_max_dur is None:
        relabel_max_dur = min_duration
    fixed, _, _ = fix_overlaps(segments)
    filtered, _ = filter_short_segments(fixed, min_duration)
    current = filtered
    for _ in range(5):
        relabeled, rc = relabel_isolated_segments(current, max_duration=relabel_max_dur, max_gap=relabel_max_gap)
        merged, mc = merge_same_speaker(relabeled, merge_gap)
        if rc == 0 and mc == 0:
            break
        current = merged
    return current


# ── Diarize a single file ───────────────────────────────────────────

def diarize_one(audio_path, use_two_pass=True):
    """Run pipeline on one file, return raw segment dicts."""
    waveform, sr = torchaudio.load(audio_path)
    audio_in = {"waveform": waveform, "sample_rate": sr}

    if use_two_pass:
        out1 = pipeline(audio_in)
        n_spk = len(set(s for _, s in out1.speaker_diarization))
        output = pipeline(audio_in, num_speakers=n_spk)
    else:
        output = pipeline(audio_in)

    label_map, nxt = {}, 1
    segs = []
    for turn, spk in output.speaker_diarization:
        if spk not in label_map:
            label_map[spk] = f"SPEAKER_{nxt}"; nxt += 1
        segs.append({"start_time": round(turn.start, 2),
                      "end_time": round(turn.end, 2),
                      "speaker_id": label_map[spk]})
    return segs


def compute_der(pred_segments, ref_annotation, collar=COLLAR):
    """Compute DER for one file. Returns (DER, components dict)."""
    pred_ann = segments_to_pyannote(pred_segments, uri=ref_annotation.uri)
    metric = DiarizationErrorRate(collar=collar, skip_overlap=False)
    der = metric(ref_annotation, pred_ann)
    # Get components via compute_components (not metric[uri])
    components = metric.compute_components(ref_annotation, pred_ann)
    return der, components


print("Helper functions ready")

## Pre-compute Raw Diarization (no post-processing)

Run the pipeline once per file to get raw predictions.
We cache these so we can sweep post-processing params without re-running inference.

**Important:** For threshold sweep, we must re-run the pipeline per threshold.
But for post-processing sweep, we reuse raw outputs from the best threshold.

## Phase 1: Clustering Threshold Sweep

This is the single most impactful parameter.

**Timing optimization:** We use **single-pass** diarization during the sweep
(phase 1 + 1b) to cut runtime in half. Two-pass is only used in Phase 2 when
we cache final predictions at the best threshold.

Estimated time: **~2-3 hours total** for all phases on Kaggle T4.

In [ ]:
# === Phase 1: Clustering Threshold Sweep ===
#
# Optimized: Single-pass diarization to save time during sweep.
# Two-pass is only used in Phase 2 for final predictions.
# Coarse grid: 10 thresholds × 1 mdo value first, then test mdo separately.

THRESHOLDS_COARSE = [0.40, 0.45, 0.50, 0.55, 0.58, 0.60, 0.62, 0.65, 0.70, 0.80]

# Preload waveforms to avoid repeated disk I/O
print("Preloading dev waveforms …")
dev_waveforms = {}
for stem in tqdm(dev_stems, desc="Loading"):
    vocal_path = VOCALS_DIR / f"{stem}.wav"
    waveform, sr = torchaudio.load(vocal_path)
    dev_waveforms[stem] = {"waveform": waveform, "sample_rate": sr}
print(f"Loaded {len(dev_waveforms)} waveforms into memory")

# === Helper: single-pass diarize + DER ===
def diarize_single_pass(pipeline, audio_in):
    """Single-pass diarization (faster, used for sweep)."""
    output = pipeline(audio_in)
    label_map, nxt = {}, 1
    segs = []
    for turn, spk in output.speaker_diarization:
        if spk not in label_map:
            label_map[spk] = f"SPEAKER_{nxt}"; nxt += 1
        segs.append({"start_time": round(turn.start, 2),
                      "end_time": round(turn.end, 2),
                      "speaker_id": label_map[spk]})
    return segs


def diarize_two_pass(pipeline, audio_in):
    """Two-pass diarization (more accurate, used for final predictions)."""
    out1 = pipeline(audio_in)
    n_spk = len(set(s for _, s in out1.speaker_diarization))
    output = pipeline(audio_in, num_speakers=n_spk)
    label_map, nxt = {}, 1
    segs = []
    for turn, spk in output.speaker_diarization:
        if spk not in label_map:
            label_map[spk] = f"SPEAKER_{nxt}"; nxt += 1
        segs.append({"start_time": round(turn.start, 2),
                      "end_time": round(turn.end, 2),
                      "speaker_id": label_map[spk]})
    return segs


# Run coarse sweep (single-pass, mdo=0.0 first)
results_phase1 = []
import time as _time
t0 = _time.time()

for thresh in THRESHOLDS_COARSE:
    pipeline.instantiate({
        "clustering": {"threshold": thresh},
        "segmentation": {"min_duration_off": 0.0},
    })

    file_ders = []
    for stem in dev_stems:
        segs = diarize_single_pass(pipeline, dev_waveforms[stem])
        der, _ = compute_der(segs, ground_truth[stem])
        file_ders.append(der)

    mean_der = np.mean(file_ders)
    median_der = np.median(file_ders)
    results_phase1.append({
        "threshold": thresh, "min_duration_off": 0.0,
        "mean_der": mean_der, "median_der": median_der,
        "std_der": np.std(file_ders), "min_der": np.min(file_ders), "max_der": np.max(file_ders),
    })
    elapsed = _time.time() - t0
    print(f"  thresh={thresh:.2f}  →  mean_DER={mean_der:.4f}  median={median_der:.4f}  [{elapsed:.0f}s elapsed]")

    gc.collect(); torch.cuda.empty_cache()

# Also test mdo=0.3 at the top 3 thresholds
df1_temp = pd.DataFrame(results_phase1).sort_values("mean_der")
top3_thresholds = df1_temp.head(3)["threshold"].tolist()
print(f"\nTesting min_duration_off=0.3 at top 3 thresholds: {top3_thresholds}")

for thresh in top3_thresholds:
    pipeline.instantiate({
        "clustering": {"threshold": thresh},
        "segmentation": {"min_duration_off": 0.3},
    })

    file_ders = []
    for stem in dev_stems:
        segs = diarize_single_pass(pipeline, dev_waveforms[stem])
        der, _ = compute_der(segs, ground_truth[stem])
        file_ders.append(der)

    mean_der = np.mean(file_ders)
    results_phase1.append({
        "threshold": thresh, "min_duration_off": 0.3,
        "mean_der": mean_der, "median_der": np.median(file_ders),
        "std_der": np.std(file_ders), "min_der": np.min(file_ders), "max_der": np.max(file_ders),
    })
    elapsed = _time.time() - t0
    print(f"  thresh={thresh:.2f}  mdo=0.3  →  mean_DER={mean_der:.4f}  [{elapsed:.0f}s elapsed]")

    gc.collect(); torch.cuda.empty_cache()

df1 = pd.DataFrame(results_phase1).sort_values("mean_der")
print(f"\n=== Phase 1 Results (sorted by mean DER) ===  [total: {_time.time()-t0:.0f}s]")
print(df1.to_string(index=False))

In [ ]:
# === Phase 1b: Fine-grained threshold sweep around the best coarse value ===

best_row = df1.iloc[0]
best_thresh_coarse = best_row["threshold"]
best_mdo = best_row["min_duration_off"]
print(f"Best coarse: threshold={best_thresh_coarse:.2f}, min_duration_off={best_mdo:.1f}, DER={best_row['mean_der']:.4f}")

# Fine sweep: ±0.04 around best in steps of 0.01 (single-pass)
THRESHOLDS_FINE = np.arange(
    max(0.35, best_thresh_coarse - 0.04),
    min(0.85, best_thresh_coarse + 0.05),
    0.01
)

results_phase1b = []
t0b = _time.time()

for thresh in THRESHOLDS_FINE:
    pipeline.instantiate({
        "clustering": {"threshold": float(thresh)},
        "segmentation": {"min_duration_off": best_mdo},
    })

    file_ders = []
    for stem in dev_stems:
        segs = diarize_single_pass(pipeline, dev_waveforms[stem])
        der, _ = compute_der(segs, ground_truth[stem])
        file_ders.append(der)

    mean_der = np.mean(file_ders)
    results_phase1b.append({
        "threshold": float(thresh), "min_duration_off": best_mdo,
        "mean_der": mean_der, "median_der": np.median(file_ders),
        "std_der": np.std(file_ders),
    })
    print(f"  thresh={thresh:.2f}  →  mean_DER={mean_der:.4f}")

    gc.collect(); torch.cuda.empty_cache()

df1b = pd.DataFrame(results_phase1b).sort_values("mean_der")
print(f"\n=== Phase 1b Fine Results ===  [{_time.time()-t0b:.0f}s]")
print(df1b.head(10).to_string(index=False))

# Select best threshold
BEST_THRESHOLD = df1b.iloc[0]["threshold"]
BEST_MDO = df1b.iloc[0]["min_duration_off"]
BEST_DER_RAW = df1b.iloc[0]["mean_der"]
print(f"\n★ Best threshold: {BEST_THRESHOLD:.2f}  (min_duration_off={BEST_MDO:.1f})  →  raw DER={BEST_DER_RAW:.4f}")

## Phase 2: Cache Raw Predictions at Best Threshold

Now that we have the best clustering threshold, run inference once and cache
the raw (no post-processing) segment predictions. Phase 3 sweeps post-processing
params offline (no GPU needed).

In [ ]:
# === Cache raw predictions at the best threshold (TWO-PASS for accuracy) ===

pipeline.instantiate({
    "clustering": {"threshold": BEST_THRESHOLD},
    "segmentation": {"min_duration_off": BEST_MDO},
})

raw_predictions = {}  # stem → list of segment dicts
t0c = _time.time()

for stem in tqdm(dev_stems, desc="Caching preds (two-pass)"):
    segs = diarize_two_pass(pipeline, dev_waveforms[stem])
    raw_predictions[stem] = segs

# Verify: DER without post-processing
ders_raw = []
for stem in dev_stems:
    der, _ = compute_der(raw_predictions[stem], ground_truth[stem])
    ders_raw.append(der)
print(f"Raw DER at best threshold ({BEST_THRESHOLD:.2f}, two-pass): mean={np.mean(ders_raw):.4f}, median={np.median(ders_raw):.4f}")
print(f"Cache time: {_time.time()-t0c:.0f}s")

# Free waveforms — no longer needed for Phase 3
del dev_waveforms
gc.collect(); torch.cuda.empty_cache()
print("Waveforms freed")

## Phase 3: Post-Processing Parameter Sweep

Sweep `min_duration`, `merge_gap`, and `relabel_max_gap` on cached raw predictions.
This is CPU-only and very fast — no pipeline re-runs needed.

In [ ]:
# === Phase 3: Post-processing grid search (CPU-only, fast) ===

MIN_DURATIONS   = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
MERGE_GAPS      = [0.0, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
RELABEL_MAX_GAPS = [0.0, 0.20, 0.30, 0.40, 0.50]  # 0.0 = disable relabeling

results_phase3 = []
total_combos = len(MIN_DURATIONS) * len(MERGE_GAPS) * len(RELABEL_MAX_GAPS)
print(f"Sweeping {total_combos} combinations …\n")

for min_dur in MIN_DURATIONS:
    for mg in MERGE_GAPS:
        for rlg in RELABEL_MAX_GAPS:
            file_ders = []
            for stem in dev_stems:
                raw = raw_predictions[stem]
                # Apply post-processing
                if rlg == 0.0 and min_dur == 0.0 and mg == 0.0:
                    processed = raw  # no-op
                else:
                    processed = postprocess(
                        raw,
                        min_duration=min_dur,
                        merge_gap=mg,
                        relabel_max_dur=max(min_dur, 0.05),  # relabel threshold tracks min_dur
                        relabel_max_gap=rlg,
                    )
                der, _ = compute_der(processed, ground_truth[stem])
                file_ders.append(der)

            mean_der = np.mean(file_ders)
            results_phase3.append({
                "min_duration": min_dur, "merge_gap": mg,
                "relabel_max_gap": rlg,
                "mean_der": mean_der,
                "median_der": np.median(file_ders),
                "std_der": np.std(file_ders),
            })

df3 = pd.DataFrame(results_phase3).sort_values("mean_der")
print("=== Phase 3 Top 20 (sorted by mean DER) ===")
print(df3.head(20).to_string(index=False))

# Best overall
best_pp = df3.iloc[0]
print(f"\n★ Best post-processing:")
print(f"  min_duration   = {best_pp['min_duration']}")
print(f"  merge_gap      = {best_pp['merge_gap']}")
print(f"  relabel_max_gap= {best_pp['relabel_max_gap']}")
print(f"  mean DER       = {best_pp['mean_der']:.4f}")
print(f"  (vs raw DER    = {BEST_DER_RAW:.4f}  →  Δ = {best_pp['mean_der'] - BEST_DER_RAW:+.4f})")

## Summary & Optimal Configuration

Print the final optimal config to paste into inference notebook.

In [ ]:
# === Final Summary ===

print("=" * 70)
print("OPTIMAL CONFIGURATION FOR INFERENCE NOTEBOOK")
print("=" * 70)
print(f"""
# ── Paste into inference notebook ──

# Pipeline hyperparameters
pipeline.instantiate({{
    "clustering": {{
        "threshold": {BEST_THRESHOLD:.2f},
    }},
    "segmentation": {{
        "min_duration_off": {BEST_MDO:.1f},
    }}
}})

# Post-processing parameters
MIN_DURATION = {best_pp['min_duration']}
MERGE_GAP    = {best_pp['merge_gap']}
RELABEL_MAX_GAP = {best_pp['relabel_max_gap']}

# DER improvement:
#   Your current:    0.192  (threshold=0.62, min_dur=0.15, merge_gap=0.20)
#   Raw (no PP):     {BEST_DER_RAW:.4f}  (threshold={BEST_THRESHOLD:.2f})
#   With optimal PP: {best_pp['mean_der']:.4f}
""")

# Per-file DER breakdown at optimal settings
print("\n=== Per-File DER at Optimal Settings ===")
for stem in dev_stems:
    raw = raw_predictions[stem]
    processed = postprocess(
        raw,
        min_duration=best_pp['min_duration'],
        merge_gap=best_pp['merge_gap'],
        relabel_max_dur=max(best_pp['min_duration'], 0.05),
        relabel_max_gap=best_pp['relabel_max_gap'],
    )
    der_raw, _ = compute_der(raw, ground_truth[stem])
    der_pp, comp = compute_der(processed, ground_truth[stem])
    n_spk_ref = len(ground_truth[stem].labels())
    n_spk_pred = len(set(s['speaker_id'] for s in processed))
    print(f"  {stem}:  DER={der_pp:.3f}  (raw={der_raw:.3f})  spk: ref={n_spk_ref} pred={n_spk_pred}")